# S11 · De las fotografías a evidencia verificable

**Universidad Central · Big Data · 60 minutos de demostración + ampliación autónoma en el mismo cuaderno**

Laura investiga **30 actas SECOP**, **62 fotografías de dos conjuntos** y **30 voces humanas MINDS-14**. Para explicar cada herramienta empezamos con una muestra pequeña y después ampliamos a **122 fuentes**, todo aquí, sin cambiar de cuaderno. Queremos responder qué se lee, qué se observa y qué se escucha, sin confundir herramientas ni inventar relaciones entre fuentes.

1. ¿Cuál es el VALOR DEL CONTRATO, la DURACIÓN y la FORMA DE PAGO en el acta 728704921?
2. ¿Qué letras aparecen en los anuncios de una fotografía?
3. ¿Qué fotografía se recupera con la consulta «viviendas en una ladera»?
4. ¿Qué dijo una voz humana y cómo lo comprobamos contra el audio?
5. ¿Cómo preservamos archivo, modalidad y huella SHA-256?

**Lo importante es aprender a escribir código**, no ejecutar un programa terminado. **Este es el único cuaderno de la sesión:** FASE 1 (fundamentos) y FASE 2 (análisis por lotes). Cada celda introduce pocas líneas; debes editar variables, probar otras imágenes y explicar un resultado o error. Puedes borrar una celda y reescribirla.

**Fuentes separadas:** las fotos periodísticas no prueban nada sobre las actas; la voz MINDS-14 es bancaria y NO pertenece a las noticias. Las actas pueden contener datos personales; evita publicar OCR completo de nombres o números de cuenta.

## 1 · Entorno y archivos (0–6 min)

**Qué es cada herramienta:** Python ejecuta nuestras instrucciones; Pillow lee píxeles; Path representa rutas de archivos; Tesseract (instalado en Linux) reconoce caracteres. La instalación manual se explica en `INSTALACION_UBUNTU.md`. En Codespaces ya está el programa y sus paquetes, pero debes comprobarlos.

**En la terminal:** `python --version` y `tesseract --list-langs`. Debe aparecer `spa`. Luego escribe estas líneas en una celda vacía.

**Tu cambio:** sustituye la foto `006` por `009` y observa la diferencia.

In [ ]:
from pathlib import Path
from PIL import Image
from IPython.display import display, Audio
import hashlib, io, json, re

RAIZ = Path.cwd()
CONFIG_S11 = json.loads((RAIZ / "configuracion_s11.json").read_text(encoding="utf-8"))
assert CONFIG_S11["version"] == 1
FOTOS = RAIZ / "datos/raw/fotos"
ACTAS = RAIZ / "datos/raw/actas"
SALIDAS = RAIZ / "outputs"
SALIDAS.mkdir(exist_ok=True)

def sha256(ruta):
    return hashlib.sha256(Path(ruta).read_bytes()).hexdigest()

foto = FOTOS / "eltiempo_006.webp"
imagen = Image.open(foto)
print("Formato:", imagen.format, "Píxeles:", imagen.size)
display(imagen)

### Comprueba tu entorno antes de realizar inferencia

**Ruta de aprendizaje:** la presentación plantea preguntas y explica cada herramienta; el cuaderno permite escribir y modificar código; el Codespace reúne las dependencias y el visor privado. **No hay un segundo cuaderno**.

**¿Qué hace cada dependencia?**
- **Tesseract:** motor OCR del sistema; `pytesseract` permite usarlo desde Python.
- **Pillow:** abre imágenes, cambia tamaño y crea recortes.
- **Poppler:** consulta páginas del PDF y las convierte temporalmente en imágenes.
- **FFmpeg / SoundFile:** inspeccionan y decodifican sonido.
- **Faster-Whisper:** modelo que convierte señal acústica en texto.
- **Ollama + EmbeddingGemma 2:440m:** ejecuta búsqueda multimodal local.
- **pandas + PyArrow:** construyen y reabren el catálogo Parquet.
- **Streamlit:** visor de fuentes, no reemplaza aprender Python.

**Ejercicio breve:** abre el explorador de VS Code y localiza `datos/raw/actas_pdf`, `datos/raw/actas` y `datos/raw/fotos`. ¿Por qué una página PNG no cuenta como otro contrato?

**EDITA AQUÍ:** en la celda siguiente cambia la fotografía de prueba. La comprobación es ligera, sin modelos ni descargas.

In [ ]:
import shutil, importlib.util
MUESTRA_FOTO = "eltiempo_023.webp"  # EDITA AQUÍ: prueba eltiempo_006.webp
for nombre in ("tesseract", "ffmpeg", "pdfinfo", "ollama"):
    print(nombre, "→", shutil.which(nombre) or "No se encuentra")
for modulo in ("pytesseract", "pandas", "pyarrow", "faster_whisper", "requests"):
    print(modulo, "→", "OK" if importlib.util.find_spec(modulo) else "Falta instalar")
datos_cargados = {
    "fotos originales": list(FOTOS.glob("eltiempo_*")),
    "fotos con ficha": list((RAIZ/"datos/raw/fotos_catalogadas").glob("*.webp")),
    "PDF originales": list((RAIZ/"datos/raw/actas_pdf").glob("*.pdf")),
    "PNG derivados": list(ACTAS.glob("*_pagina1.png"))
}
for tipo, archivos in datos_cargados.items():
    print(f"{tipo}: {len(archivos)}")
assert [len(a) for a in datos_cargados.values()] == [30, 32, 30, 30]
display(Image.open(FOTOS / MUESTRA_FOTO))

### Glosario para que puedas reproducirlo en otro proyecto

| Término | Qué significa | Qué NO significa |
|---|---|---|
| **RAW** | archivo original conservado sin alteraciones | una predicción ya verificada |
| **Derivado** | imagen, WAV o tabla producida desde el original | una fuente adicional |
| **OCR** | convertir píxeles de caracteres a texto probable | describir toda la escena |
| **PSM** | modo de segmentación de página de Tesseract | porcentaje de confianza |
| **ASR** | transcribir voz a texto probable | certeza de cada palabra |
| **Embedding** | representación numérica del contenido | descripción redactada ni prueba de verdad |
| **Coseno** | cercanía geométrica entre vectores | probabilidad calibrada |
| **ETL** | extracción, transformación y carga de datos | necesariamente computación distribuida |
| **SHA-256** | huella de bytes para detectar cambios | certificado de autenticidad |
| **Procedencia** | ruta, fuente, licencia y transformación | solo un nombre descriptivo |

**Transferencia:** cada técnica debe funcionar con otra carpeta y otra pregunta. Las variables marcadas **EDITA AQUÍ** son el punto de partida; explica qué cambio hiciste y qué resultado observaste.

## 2 · OCR documental real: un acta de inicio SECOP (6–18 min)

**Pregunta de investigación:** ¿cuál es el valor contractual, cuánto dura y cómo se pagará? Trabajamos con el documento original **secop_acta_728704921.pdf** (consulta sus páginas en el visor).

**OCR** significa reconocimiento óptico de caracteres: Tesseract intenta convertir **píxeles** en palabras. **pytesseract** permite llamarlo desde Python. Un documento con tablas puede confundir el orden de lectura: una frase de una columna puede mezclarse con la etiqueta de otra.

**PSM** (*Page Segmentation Mode*, modo de segmentación de página) indica cómo buscar bloques de texto: **3** detecta el diseño automáticamente (punto de partida para esta página), **6** supone un bloque de texto y **11** intenta encontrar fragmentos dispersos. **No es un porcentaje ni una puntuación de calidad**.

**En pantalla:** muestra el PDF original, selecciona su primera página, ubica los campos VALOR DEL CONTRATO, DURACIÓN y FORMA DE PAGO; después compara con los resultados del OCR completo en el visor.

**Privacidad:** las actas pueden incluir nombres e identificaciones. Para grabar usa un recorte que destaque campos contractuales y no publiques la transcripción completa sin revisarla.

In [ ]:
import pytesseract

# El PDF original permanece intacto; para este ejemplo usamos su primera página PNG.
acta_pdf = RAIZ / "datos/raw/actas_pdf/secop_acta_728704921.pdf"
acta = ACTAS / "secop_acta_728704921_pagina1.png"
pagina = Image.open(acta)
print("Archivo:", acta_pdf.name, "| Imagen:", pagina.size)

# Acercamiento didáctico a las filas de valor y forma de pago.
recorte = pagina.crop((int(pagina.width*0.12), int(pagina.height*0.39),
                       int(pagina.width*0.91), int(pagina.height*0.64)))
display(recorte.resize((780, int(recorte.height*780/recorte.width))))

ocr_3 = pytesseract.image_to_string(pagina, lang="spa", config="--psm 3")
print("Caracteres OCR:", len(ocr_3), "| Líneas:", len(ocr_3.splitlines()))
print("Revisa el texto completo con privacidad en la pestaña OCR del visor.")
for linea in ocr_3.splitlines():
    if any(campo in linea.upper() for campo in
           ("VALOR DEL CONTRATO", "FORMA DE PAGO", "DURACI", "ACTA DE INICIO")):
        print("Candidato:", linea.strip()[:160])

### Verificación independiente: no confundir salida OCR y evidencia visual

En la **primera página original** se observan, mediante lectura humana:

| Campo | Referencia visual del PDF |
|---|---|
| Tipo | Acta de inicio |
| Contrato | 006-2026 |
| Valor contractual | **$14.760.000 COP** |
| Plazo | **6 meses** |
| Forma de pago | **6 pagos de $2.460.000 COP** |

**Importante:** estos valores son la **referencia visual para cotejo**, **NO** una supuesta predicción del motor. Puede ocurrir que el OCR encuentre palabras de una columna y las ubique incorrectamente junto a otra etiqueta.

**Comprobación aritmética independiente:** 6 × 2.460.000 = 14.760.000. Esto verifica coherencia matemática entre cifras observadas, **no certifica autenticidad de la contratación**.

**Reto:** observa y explica si Tesseract reconoció ambas cantidades, si detectó el número de contrato y si confundió letras o columnas. Prueba PSM 3, 6 y 11 sobre **el mismo archivo**.

### Flujo de una lectura documental válida

**PDF original → seleccionar página → imagen → segmentación PSM → texto OCR → campo candidato → cotejo humano en el original**.

Trabajamos con `secop_acta_728704921.pdf`. En la primera página pueden observarse el contrato 006-2026, su valor de **$14.760.000**, una duración de **6 meses** y **seis pagos de $2.460.000**. Son **referencias visuales**, no una salida inventada por el programa.

**Tres capas que no debes mezclar:**
1. **Fuente original:** la página real que el visor permite ampliar.
2. **Predicción automática:** la cadena devuelta por Tesseract. Puede mezclar columnas, perder acentos u omitir números.
3. **Interpretación:** la etiqueta y el importe que tú propones tras examinar el fragmento y comprobarlo en la página.

**EDITA AQUÍ:** cambia PSM 3 a PSM 6; mantén la misma página y el idioma. Un OCR con más caracteres no necesariamente es más exacto.

In [ ]:
# Variamos solo el PSM; el documento y el idioma siguen siendo iguales.
modos = [3, 6, 11]
for psm in modos:
    texto_psm = pytesseract.image_to_string(
        pagina, lang="spa", config=f"--psm {psm}")
    lineas_utiles = [linea.strip()[:140] for linea in texto_psm.splitlines()
                     if any(x in linea.upper() for x in
                            ("VALOR DEL CONTRATO", "FORMA DE PAGO", "DURACI"))]
    print(f"PSM {psm}: {len(texto_psm)} caracteres · {len(lineas_utiles)} líneas candidatas")
    print(lineas_utiles[:3] or ["No se recuperó ninguna etiqueta buscada"])

# Referencia OBSERVADA EN LA FUENTE, no predicción de OCR:
valor_visual = 14_760_000
cuotas_visual = 6
pago_visual = 2_460_000
print("¿Coincide el producto?", cuotas_visual * pago_visual == valor_visual)

### Extraer campos: primero encontrar evidencia, después interpretar

Para extraer un valor no basta encontrar cualquier número. Debemos localizar la etiqueta **VALOR DEL CONTRATO** y examinar el texto cercano. Escribe una función corta que devuelva un **fragmento de OCR sin corregir**; si no reconoce la etiqueta, reporta que no la encontró. **No sustituyas silenciosamente un texto fallido por los valores de la referencia visual.**

In [ ]:
def fragmento_cercano(texto, etiqueta, caracteres=220):
    coincidencia = re.search(etiqueta, texto, flags=re.IGNORECASE)
    if not coincidencia:
        return None
    return texto[coincidencia.start():coincidencia.start()+caracteres]

for campo in ("VALOR DEL CONTRATO", "FORMA DE PAGO", "DURACI"):
    fragmento = fragmento_cercano(ocr_3, campo)
    print(campo, "→", "[No detectado]" if fragmento is None else fragmento[:160].replace("\n", " "))

# Cambia caracteres=220 a 400 y comprueba si se mezclan columnas.
# En el visor puedes leer el OCR completo y comparar con el PDF original.

## 3 · OCR en fotografías: por qué un logotipo puede fallar

Una persona lee en la fotografía 023 «Lotería del VALLE · Cambia tu vida», pero Tesseract puede devolver letras incompletas. ¿Por qué?

**La imagen es un diseño publicitario**, no un documento de texto continuo: las letras son curvas, de colores, tienen contornos decorativos y conviven con una persona que ocupa casi media fotografía. El OCR no sabe automáticamente qué región queremos analizar.

**Experimento paso a paso:**
1. Abre en el visor `eltiempo_023.webp` y obsérvala.
2. Ejecuta el bucle sobre cuatro fotografías con **PSM_FOTOS = 11**.
3. **EDITA AQUÍ:** cambia PSM_FOTOS a 6 y vuelve a ejecutar.
4. Examina el **texto completo**. No uses `repr(texto[:130])`, que recorta la información y muestra barras como `\n` en vez de los saltos reales.
5. Después recorta únicamente la palabra VALLE y compara el PSM 7.

**Pregunta pedagógica:** ¿por qué una función puede terminar sin errores y, aun así, producir un resultado OCR incorrecto?

In [ ]:
nombres = CONFIG_S11["muestras_fotografias"][:4]  # EDITA AQUÍ: usa otra muestra autorizada
PSM_FOTOS = 11  # EDITA AQUÍ: vuelve a ejecutar con 6
textos_fotos = {}  # Diccionario para conservar cada resultado real

for nombre in nombres:
    ruta = FOTOS / nombre
    imagen_foto = Image.open(ruta)
    leido = pytesseract.image_to_string(
        imagen_foto, lang="spa", config=f"--psm {PSM_FOTOS}"
    ).strip()
    textos_fotos[nombre] = leido
    print(f"\n{'='*15} {nombre} {'='*15}")
    print("Caracteres:", len(leido))
    print(leido if leido else "(Tesseract no reconoció palabras)")

### Laboratorio guiado · Aislar la palabra VALLE

**¿Qué debes mostrar?** La fotografía completa y, al lado, el recorte que contiene las letras de colores. La comparación evita confundir la salida incorrecta de OCR con una descripción de la imagen.

**¿Qué significan las instrucciones?**
- `Image.open` lee la foto y `size` proporciona ancho y alto.
- `crop` necesita cuatro coordenadas (izquierda, arriba, derecha, abajo).
- `PSM 7` indica a Tesseract que el recorte tiene **una sola línea**. `PSM 6` supone un bloque uniforme y `PSM 11` busca texto disperso.
- `print` enseña **la respuesta real del motor**, sin reemplazarla por la palabra esperada.

**EDITA AQUÍ:** cambia `ZONA_VALLE` y después `MODOS_LOGO`. El primer recorte es una propuesta para esta foto; prueba otros y explica cuál fue mejor.

**Referencia humana:** la palabra que vemos es VALLE. Si OCR devuelve «VALE», «(VALLE)» o algo incompleto, eso se registra como predicción, no como verdad.

In [ ]:
ruta_logo = FOTOS / "eltiempo_023.webp"  # EDITA AQUÍ: foto fuente
foto_logo = Image.open(ruta_logo).convert("RGB")
ancho, alto = foto_logo.size
print("Dimensiones:", ancho, "×", alto)

# EDITA AQUÍ: cuatro límites proporcionales; 0 es inicio y 1 es extremo.
ZONA_VALLE = (175/1200, 240/675, 540/1200, 375/675)
caja = (round(ancho*ZONA_VALLE[0]), round(alto*ZONA_VALLE[1]),
        round(ancho*ZONA_VALLE[2]), round(alto*ZONA_VALLE[3]))
region_valle = foto_logo.crop(caja)
display(foto_logo)
display(region_valle)

MODOS_LOGO = [6, 7, 11]  # EDITA AQUÍ: compara PSM
for psm_logo in MODOS_LOGO:
    salida = pytesseract.image_to_string(
        region_valle, lang="spa", config=f"--psm {psm_logo}"
    ).strip()
    print(f"PSM {psm_logo}:", salida or "(no reconoció texto)")

### Experimento controlado · ¿Ampliar la imagen mejora OCR?

**¿Por qué un aumento podría ayudar?** Las letras tienen pocos píxeles y contornos curvos. `resize` con `LANCZOS` interpola nuevos píxeles y puede facilitar la segmentación, aunque **no recupera información ausente**.

Primero mantén el recorte y PSM 7 constantes, y cambia **solo FACTOR = 1 → 2**. Después mantén la escala y prueba **USAR_CONTRASTE = True**.

**¿Qué hace cada función?** `ImageOps.autocontrast` redistribuye intensidades; `ImageFilter.UnsharpMask` resalta bordes; `image_to_string` devuelve la predicción OCR. Estas transformaciones pueden mejorar o empeorar resultados, así que necesitamos comparar, no suponer.

**EDITA AQUÍ:** FACTOR, USAR_CONTRASTE y PSM_RECORTADO.

In [ ]:
from PIL import ImageOps, ImageFilter

FACTOR = 2             # EDITA AQUÍ: probar 1, 2 o 3
USAR_CONTRASTE = False # EDITA AQUÍ: probar después True
PSM_RECORTADO = 7      # Una sola línea

region_procesada = region_valle.resize(
    (region_valle.width*FACTOR, region_valle.height*FACTOR),
    Image.Resampling.LANCZOS
)
if USAR_CONTRASTE:
    region_procesada = ImageOps.autocontrast(
        ImageOps.grayscale(region_procesada))
    region_procesada = region_procesada.filter(
        ImageFilter.UnsharpMask(radius=1, percent=130, threshold=2))
display(region_procesada)

salida_real = pytesseract.image_to_string(
    region_procesada, lang="spa", config=f"--psm {PSM_RECORTADO}"
).strip()
print("OCR real:", salida_real or "(no detectó texto)")
print("Referencia humana: VALLE (no es predicción OCR)")

### Evaluar la salida: texto, posición y confianza interna

`image_to_data` devuelve palabras con sus coordenadas y una confianza estimada por Tesseract. Esto muestra **qué parte del recorte interpretó como texto**.

Una puntuación interna de 90 **no es un 90 % de exactitud real**. Para medir exactitud necesitamos una referencia humana independiente y una métrica de error.

**EDITA AQUÍ:** cambia CONFIANZA_MINIMA de 0 a 40, comprueba qué palabras se omiten y mira si eran correctas en la imagen.

In [ ]:
datos_ocr = pytesseract.image_to_data(
    region_procesada, lang="spa", config=f"--psm {PSM_RECORTADO}",
    output_type=pytesseract.Output.DICT)
CONFIANZA_MINIMA = 0  # EDITA AQUÍ: prueba 40
for i, palabra in enumerate(datos_ocr["text"]):
    if not palabra.strip():
        continue
    confianza = float(datos_ocr["conf"][i])
    if confianza >= CONFIANZA_MINIMA:
        print("Palabra:", palabra, "| confianza interna:", round(confianza, 1),
              "| posición:", (datos_ocr["left"][i], datos_ocr["top"][i]))

### Mini experimento controlado: la palabra VALLE

La fotografía de la Lotería del Valle combina letras curvas, colores, contornos decorativos y una persona. **Una persona puede leer el logotipo aunque Tesseract se equivoque.**

| Variante | Entrada | PSM | Pregunta |
|---|---|---:|---|
| A | foto original completa | 11 | ¿cuánto texto reconoce sin indicarle la región? |
| B | recorte de la palabra VALLE | 7 | ¿mejora al aislar una línea? |
| C | mismo recorte ampliado ×2 | 7 | ¿ayuda la escala, manteniendo el resto igual? |
| D | mismo recorte ampliado con contraste | 7 | ¿la transformación mejora o deteriora el resultado? |

**Qué enseñar:** mantén constantes la región y el PSM entre B y C para atribuir cualquier cambio al tamaño; entre C y D cambia solo el contraste.

**Qué editar:** `ZONA_VALLE`, `MODOS_LOGO`, `FACTOR` y `USAR_CONTRASTE`. El texto de referencia leído por una persona **jamás se copia como salida del OCR**.

**Criterio de éxito:** identificar una predicción correcta, un falso positivo y explicar por qué la cantidad de caracteres o la confianza interna no bastan para evaluar exactitud.

## 4 · Búsqueda por significado con EmbeddingGemma 2 (25–35 min)

**Pregunta:** ¿cuál fotografía muestra viviendas en una ladera? OCR reconoce letras; un **embedding** representa contenido en un vector de **768 números**.

**Herramientas:** Ollama ejecuta `embeddinggemma-2:440m` en un servidor local. La API `POST /api/embed` recibe **`input` como lista**: para una fotografía usamos `{"input": [{"image": "<bytes base64>"}]}`, para texto `{"input": ["pregunta"]}`. El elemento `[0]` de `embeddings` es el vector correspondiente a la única entrada. **Base64** transporta bytes dentro de JSON: no es inteligencia artificial. Pillow abre y normaliza WebP/JPG a JPEG RGB para evitar diferencias de formato sin alterar el archivo fuente.

**Preparación del Codespace:** ejecuta en Terminal `bash preparar_embeddinggemma.sh` y comprueba el listado con `ollama list`.

**Lo que haces en el código siguiente:**
1. Empiezas con los **bytes originales** del archivo (formato directo de los originales) y los conviertes a Base64. La variable `NORMALIZAR_JPEG = False` mantiene intacta esa entrada.
2. Si cambias `NORMALIZAR_JPEG = True`, Pillow convierte temporalmente a JPEG RGB y limita el lado a 768 píxeles. Esta es una comparación opcional de formatos, **no la causa ni la solución del HTTP 400**.
3. Compruebas **una sola imagen** antes de empezar las otras. Imprimes el nombre de archivo.
4. Ante HTTP 400, lees el **mensaje real del servidor**, no solo `raise_for_status()`.
5. Verificas que el resultado tenga 768 dimensiones.

**EDITA AQUÍ:** cambia `primer_nombre` para elegir otra fotografía. Para estudiar la conversión, primero activa `NORMALIZAR_JPEG = True` y **después** modifica `MAX_LADO = 768` → 512; con `False` se envían los bytes originales y `MAX_LADO` no se aplica. Observa tamaño Base64, tiempo y resultado. La búsqueda por similitud no certifica hechos.

**Forma probada de la petición:**

```python
payload = {"model": "embeddinggemma-2:440m", "input": [{"image": base64_imagen}]}
respuesta = requests.post(API, json=payload, timeout=240)
```

**EDITA AQUÍ:** en la función `vector` busca `"input": [entrada]`. Los **corchetes exteriores son importantes**: el formato de lista funciona con una imagen o consulta y así mantenemos un formato reproducible para imagen y consulta de texto. La conversión a JPEG que aparece en otro paso es una transformación opcional para comparar, **no es la corrección del HTTP 400**.


In [ ]:
import requests, base64, math
from PIL import ImageOps
from io import BytesIO

MODELO = CONFIG_S11["modelo_visual"]  # mismo modelo que el visor
API = "http://127.0.0.1:11434/api/embed"
MAX_LADO = 768  # EDITA AQUÍ: prueba 512 solo cuando NORMALIZAR_JPEG = True

NORMALIZAR_JPEG = False  # EDITA AQUÍ: False = bytes originales (probado); True = JPEG uniforme

def preparar_imagen_base64(ruta):
    """Base64 de bytes originales; opcionalmente convierte a JPEG RGB para experimentar."""
    if not NORMALIZAR_JPEG:
        return base64.b64encode(ruta.read_bytes()).decode("ascii")
    with Image.open(ruta) as original:
        imagen = ImageOps.exif_transpose(original).convert("RGB")
        imagen.thumbnail((MAX_LADO, MAX_LADO))
        memoria = BytesIO()
        imagen.save(memoria, format="JPEG", quality=85)
    return base64.b64encode(memoria.getvalue()).decode("ascii")

def vector(entrada, nombre="consulta"):
    """Enviar UN elemento dentro de input=[entrada] y mostrar errores de Ollama."""
    try:
        respuesta = requests.post(API, json={"model": MODELO, "input": [entrada]},
                                  timeout=240)
    except requests.RequestException as exc:
        raise RuntimeError("Ollama no responde. Prueba bash iniciar_ollama.sh") from exc
    if not respuesta.ok:
        raise RuntimeError(f"Ollama rechazó {nombre} (HTTP {respuesta.status_code}): "
                           f"{respuesta.text[:1500]}")
    vectores = respuesta.json().get("embeddings", [])
    if len(vectores) != 1 or len(vectores[0]) != 768:
        raise RuntimeError(f"No se obtuvo un vector de 768 dimensiones para {nombre}")
    return vectores[0]

# PREVIO: verificamos una fotografía antes del bucle.
primer_nombre = nombres[0]  # EDITA AQUÍ: también prueba eltiempo_023.webp
contenido = preparar_imagen_base64(FOTOS / primer_nombre)
print("Primera foto:", primer_nombre, "longitud Base64:", len(contenido))
primer_vector = vector({"image": contenido}, primer_nombre)
print("Comprobación correcta:", len(primer_vector), "dimensiones")

vectores_fotos = {primer_nombre: primer_vector}
for nombre in nombres[1:]:
    print("Procesando", nombre)
    vectores_fotos[nombre] = vector(
        {"image": preparar_imagen_base64(FOTOS / nombre)}, nombre
    )
    print("Vector obtenido:", len(vectores_fotos[nombre]), "valores")
assert len(vectores_fotos) == 4

### Del archivo al vector: explicación línea por línea

La llamada a Ollama usa la API `/api/embed`, con un **objeto de imagen dentro de una lista**, incluso si solo enviamos una fotografía:

~~~python
{"model": "embeddinggemma-2:440m", "input": [{"image": base64_imagen}]}
~~~

**Qué hace cada parte:**
- `read_bytes()` recupera los bytes reales de la foto; `base64.b64encode` los convierte a una cadena compatible con JSON.
- `requests.post` envía esa cadena al servicio local de Ollama. `timeout` limita la espera.
- `input: [entrada]` es la estructura **verificada en Codespaces** para texto e imagen.
- `embeddings[0]` obtiene el vector del primer elemento enviado. `len(...)=768` comprueba su tamaño, **no su calidad**.
- `NORMALIZAR_JPEG=False` conserva la foto original; con `True` estudias una conversión opcional, sin sobrescribir RAW.
- Si el servidor devuelve HTTP 400, el código imprime **su respuesta**, no únicamente la excepción de Python.

**EDITA AQUÍ:** usa una fotografía diferente, luego cambia la pregunta. Primero prueba **una** foto, después cuatro; las 62 se procesan solo cuando sea necesario y puedan reutilizarse los vectores guardados.

### Tu búsqueda: cambia la pregunta

**Conceptos:** producto punto y norma de un vector. El coseno divide el producto punto entre las normas. Este cálculo es pequeño y comprensible; no hace falta una base de datos vectorial para cuatro imágenes.

Prueba: «viviendas en una ladera», «medicamentos», «taquillas de transporte» y «sorteo de lotería». **Evalúa manualmente** si el primer resultado corresponde a la imagen, sin afirmar precisión universal.

In [ ]:
def coseno(a, b):
    producto = sum(x*y for x, y in zip(a, b))
    norma_a = math.sqrt(sum(x*x for x in a))
    norma_b = math.sqrt(sum(y*y for y in b))
    return producto / (norma_a * norma_b)

pregunta = "viviendas en una ladera"  # TU TURNO: modifica la consulta
consulta = vector("task: search result | query: " + pregunta)
ranking = sorted(((nombre, coseno(consulta, valores))
                  for nombre, valores in vectores_fotos.items()),
                 key=lambda fila: -fila[1])
for nombre, similitud in ranking:
    print(f"{similitud:.3f} · {nombre}")
display(Image.open(FOTOS / ranking[0][0]))

### Similitud coseno: medida geométrica, no porcentaje de verdad

El producto punto suma los productos de cada dimensión. Para comparar vectores, dividimos el producto punto por el producto de sus normas:

\[
\mathrm{cos}(a,b)=\frac{\sum_i a_i b_i}
{\sqrt{\sum_i a_i^2}\,\sqrt{\sum_i b_i^2}}
\]

Un resultado cercano a 1 suele indicar que los vectores apuntan en direcciones parecidas, pero **no significa «99 % de probabilidad de que la noticia sea correcta»**.

**Ejemplo pedagógico:** compara los vectores bidimensionales \(a=(1,0)\), \(b=(0.8,0.2)\) y \(c=(-1,0)\). Los vectores reales de EmbeddingGemma tienen **768 dimensiones**.

**La recuperación de información** ordena **candidatos**. No declara que el primero responda correctamente.

**Verificación independiente:** abre en el visor las tres primeras fotografías del ranking. Indica cuál sí responde a «viviendas en una ladera» y cuál no. **EDITA AQUÍ:** cambia la consulta a «lotería», «taquillas» o «medicamentos». Explica qué cambió en la clasificación.

## 5 · Transcribir voz REAL con Faster-Whisper (35–44 min)

**Fuente autorizada:** PolyAI MINDS-14, es-ES, licencia CC BY 4.0. Son conversaciones bancarias, **NO audios de las fotos de El Tiempo**.

Esta celda descarga el **Parquet** de fuente (~39 MB) desde Hugging Face, verifica la huella SHA-256 del corpus y conserva el WAV original de una persona. El audio puede contener el códec G.711: **WAV es un contenedor, no un códec**.

Antes de hablar del ASR, **escucha** el WAV. La transcripción `referencia` pertenece al corpus, no es la predicción de Whisper. Si falla la descarga o el hash, detente: no fabriques datos de sustitución.

In [ ]:
from urllib.request import Request, urlopen
import pandas as pd
import numpy as np
import soundfile as sf
import wave

URL = ("https://huggingface.co/datasets/PolyAI/minds14/resolve/main/"
       "es-ES/train-00000-of-00001.parquet")
SHA_CORPUS = "989bf2f90676a5eb5852fe785a03b1d9b97763e0febbd0fd0a3f489afd83badf"
corpus = RAIZ / "datos/minds14_es-ES.parquet"

# Solo se descarga si falta; el nombre .part evita aceptar archivos incompletos.
if not corpus.exists():
    temporal = corpus.with_suffix(".part")
    peticion = Request(URL, headers={"User-Agent": "BigData-UCentral-educativo/1.0"})
    try:
        with urlopen(peticion, timeout=180) as red, temporal.open("wb") as salida:
            while bloque := red.read(1024 * 1024):
                salida.write(bloque)
                if salida.tell() > 100_000_000:
                    raise ValueError("Descarga superior a 100 MB")
        if sha256(temporal) != SHA_CORPUS:
            raise ValueError("SHA-256 de descarga diferente de la fuente auditada")
        temporal.replace(corpus)
    finally:
        temporal.unlink(missing_ok=True)
if sha256(corpus) != SHA_CORPUS:
    raise ValueError("El Parquet de audio no coincide con la fuente")
audios = pd.read_parquet(corpus, columns=["audio", "path", "transcription"])
FILA_AUDIO = 0  # EDITA AQUÍ: cambiar a 1 o 2 y comparar voz Y referencia
fila = audios.sort_values("path", kind="stable").iloc[FILA_AUDIO]
contenido = bytes(fila["audio"]["bytes"])
origen = RAIZ / "datos/raw" / f"voz_minds14_{FILA_AUDIO:03d}_fuente.wav"
hash_esperado = hashlib.sha256(contenido).hexdigest()
if origen.exists():
    assert sha256(origen) == hash_esperado, "Fuente local diferente de la muestra seleccionada"
else:
    origen.write_bytes(contenido)  # Nunca sobrescribir un archivo RAW existente
with sf.SoundFile(io.BytesIO(contenido)) as decodificador:
    frecuencia = decodificador.samplerate
    canales = decodificador.channels
    codec = decodificador.subtype
    muestras = decodificador.read(dtype="float32", always_2d=True)
mono = muestras.mean(axis=1)
pcm16 = (np.clip(mono, -1, 1) * 32767).astype("<i2")
wav = SALIDAS / f"voz_minds14_{FILA_AUDIO:03d}_pcm16.wav"  # Derivado por muestra
with wave.open(str(wav), "wb") as destino:
    destino.setnchannels(1)
    destino.setsampwidth(2)
    destino.setframerate(frecuencia)
    destino.writeframes(pcm16.tobytes())
referencia = str(fila["transcription"])
print("Referencia del registro:", fila["path"])
print("Códec original:", codec, "| Frecuencia:", frecuencia,
      "| Canales:", canales, "| Segundos:", round(len(mono)/frecuencia, 2))
print("SHA-256 del original:", sha256(origen)[:20])
display(Audio(filename=str(wav)))

### Comprender el audio antes de transcribir

No confundas **nombre**, **contenedor**, **códec** y **señal decodificada**:
- El nombre `voz_minds14.wav` sugiere WAV, pero es preciso inspeccionar el contenido.
- **WAV** organiza los metadatos y las muestras; puede contener distintas codificaciones.
- **G.711** es una compresión de audio; **PCM16** guarda enteros de 16 bits. Convertimos solo una **copia derivada**.
- **Frecuencia de muestreo (Hz):** muestras por segundo; **canales:** mono o estéreo.
- **Duración:** número de muestras / frecuencia. No se deduce del tamaño en bytes sin conocer la codificación.

**Debes demostrar** la reproducción de la voz, el subtipo de origen, frecuencia, canales, duración, SHA del archivo original y la comparación con la **referencia del mismo registro**.

**EDITA AQUÍ:** cambia `FILA_AUDIO = 0` por `FILA_AUDIO = 1` y escucha otra voz. La transcripción humana debe cambiar con el archivo. Nunca calcules WER con referencia de una grabación diferente.

### ¿Qué aprende el modelo Whisper?

**Faster-Whisper** ejecuta `Whisper tiny` en CPU con cuantización `int8`: representación de 8 bits para reducir uso de memoria. El modelo recibe la grabación, no copia la transcripción humana. Devuelve segmentos con **inicio, fin y texto**.

**Tu verificación:** identifica una frase problemática, reproduce el intervalo y compara lo que dijo la persona con lo que predijo el modelo. No ocultes errores.

In [ ]:
from faster_whisper import WhisperModel

modelo_voz = WhisperModel("tiny", device="cpu", compute_type="int8",
                          cpu_threads=2)
segmentos, info = modelo_voz.transcribe(str(wav), language="es", beam_size=2)
segmentos = list(segmentos)
hipotesis = " ".join(s.text.strip() for s in segmentos)
print("Predicción Whisper:", hipotesis)
print("Referencia humana:", referencia)
for s in segmentos:
    print(round(s.start, 1), "→", round(s.end, 1), s.text)

### Medir WER con una función pequeña

**WER:** tasa de error de palabras = número mínimo de inserciones, borrados y sustituciones dividido entre las palabras de referencia. Aquí la referencia humana existe porque viene del corpus. Sin ella **NO** afirmes una tasa de error. El WER solo describe este audio y este método de comparación.

In [ ]:
def palabras(texto):
    return re.findall(r"\w+", texto.lower())

def errores(a, b):
    anterior = list(range(len(b) + 1))
    for i, palabra in enumerate(a, 1):
        actual = [i]
        for j, otra in enumerate(b, 1):
            actual.append(min(actual[-1] + 1, anterior[j] + 1,
                              anterior[j-1] + (palabra != otra)))
        anterior = actual
    return anterior[-1]

ref = palabras(referencia)
pred = palabras(hipotesis)
print("WER de este audio:", round(errores(ref, pred) / len(ref), 3))

### WER: una métrica que puedes calcular, explicar y cuestionar

**Word Error Rate (tasa de error de palabras)**:

\[
WER = \frac{S+D+I}{N}
\]

**S** sustituciones, **D** eliminaciones, **I** inserciones y **N** número de palabras de la referencia humana. No siempre está entre 0 y 1: puede superar 1 si hay muchas inserciones.

**Ejemplo:** referencia «valor del contrato»; predicción «valor contrato». Falta «del», entonces D=1, N=3 y WER=1/3.

**Pregunta crítica:** si no existe transcripción humana confiable, ¿podemos medir WER? **No.** Podemos escuchar el audio, revisar manualmente la hipótesis y declarar esa limitación.

**EDITA AQUÍ:** en la celda siguiente cambia la frase predicha. Identifica si agregaste, quitaste o sustituiste palabras y explica cómo cambia la métrica.

In [ ]:
referencia_prueba = "valor del contrato"
prediccion_prueba = "valor contrato"  # EDITA AQUÍ: añade o sustituye una palabra
a = palabras(referencia_prueba)
b = palabras(prediccion_prueba)
distancia = errores(a, b)
print("Referencia:", a)
print("Predicción:", b)
print("Errores mínimos:", distancia, "de", len(a), "palabras")
print("WER:", round(distancia/len(a), 3))
assert errores(palabras("valor del contrato"), palabras("valor contrato")) == 1

## 6 · ETL con Python escrito por ti (44–54 min)

**E = extraer:** abrir original, recuperar texto o voz.  
**T = transformar:** crear diccionarios comparables, conservar tipo y procedencia.  
**L = cargar:** guardar **JSONL** (un objeto por línea) y **Parquet** (columnas tipadas).

No vamos a ejecutar un motor de 400 líneas. Vas a escribir `for`, `dict`, `json.dumps` y `to_parquet`.

**Privacidad:** **no publiques el OCR completo de las actas**: pueden contener identificaciones y cuentas. Guarda solo la referencia al archivo y los metadatos mínimos. Una huella SHA-256 prueba igualdad de bytes, no que la fuente diga la verdad.

In [ ]:
registros = []
for nombre in nombres:
    archivo = FOTOS / nombre
    registros.append({"archivo": str(archivo.relative_to(RAIZ)),
                      "modalidad": "foto_periodistica",
                      "sha256": sha256(archivo),
                      "texto_recortado": textos_fotos[nombre][:100],
                      "credito": "ver ficha editorial"})

registros.append({"archivo": str(acta.relative_to(RAIZ)),
                  "modalidad": "documento_secop", "sha256": sha256(acta),
                  "texto_recortado": "[OCR protegido: datos personales]",
                  "credito": "documento público SECOP, revisar contexto"})
registros.append({"archivo": str(wav.relative_to(RAIZ)),
                  "modalidad": "audio_minds14", "sha256": sha256(wav),
                  "texto_recortado": hipotesis[:100],
                  "credito": "PolyAI MINDS-14 es-ES, CC BY 4.0"})

ruta_jsonl = SALIDAS / "catalogo.jsonl"
with ruta_jsonl.open("w", encoding="utf-8") as salida:
    for dato in registros:
        salida.write(json.dumps(dato, ensure_ascii=False) + "\n")
pd.DataFrame(registros).to_parquet(SALIDAS / "catalogo.parquet", index=False)
display(pd.DataFrame(registros)[["archivo", "modalidad", "sha256"]])

### JSON, JSONL y Parquet: tres formatos y propósitos distintos

| Formato | Estructura | Uso |
|---|---|---|
| **JSON** | objeto o lista completa en un documento | manifiestos y configuración |
| **JSONL** | **un objeto JSON por línea** | inspección registro a registro y depuración |
| **Parquet** | tabla **columnar**, con tipos | consultar columnas de un lote de datos |

Ejemplo mínimo JSONL: una línea por fuente, **sin encerrar todas las líneas en una lista**.

~~~json
{"tipo":"foto","ruta":"datos/raw/fotos/eltiempo_023.webp"}
{"tipo":"acta","ruta":"datos/raw/actas_pdf/secop_acta_728704921.pdf"}
~~~

**SHA-256** compara igualdad de **bytes**. Cambiar un único byte modifica la huella, pero **no certifica** que el texto reconocido sea verdadero o que una fuente tenga licencia abierta.

**¿Qué debe preservar el ETL?** ruta original, modalidad, atribución, SHA del original, ruta y SHA del derivado, parámetros relevantes y limitaciones.

**Tu comprobación no se limita a 122 filas:** la auditoría final compara también cada campo del JSONL con el Parquet y verifica los archivos vinculados. No exportes OCR completo de actas con datos personales.

### Verificar datos sin confiar en una salida bonita

**Comprueba:** reabrir JSONL y Parquet; contar filas; comparar los hashes actuales con los que guardaste. Una fuente alterada debe detectarse. Si quieres experimentar, modifica **una copia**, nunca el archivo RAW.

In [ ]:
filas = [json.loads(linea) for linea in ruta_jsonl.read_text().splitlines()]
tabla = pd.read_parquet(SALIDAS / "catalogo.parquet")
assert len(filas) == len(tabla)
for fila in filas:
    assert sha256(RAIZ / fila["archivo"]) == fila["sha256"]
print("OK:", len(filas), "registros, integridad de originales comprobada")

## Transición · De lo aprendido a 122 fuentes

Acabas de crear las piezas esenciales. Ahora **no cambies de cuaderno**: la siguiente sección aplica esas mismas ideas a lotes. Puedes modificar al menos tres líneas de Python:

- **Documento:** cambia a la segunda acta y compara PSM 6/11; lee visualmente un valor.
- **Imagen:** cambia pregunta y compara los cuatro resultados de búsqueda; verifica el original.
- **Audio:** toma otro registro autorizado del corpus y vuelve a transcribir, comparando la referencia humana.

**Producto breve:** `pregunta | archivo | herramienta | resultado | cómo lo verifiqué | posible error`.

**Criterio de aprendizaje:** debes poder responder **qué hace cada línea esencial**, por qué OCR no describe escenas, por qué el vector no certifica hechos, cómo se comprueba WER y cómo adaptar el flujo a otro conjunto propio. La mera ejecución de una celda no acredita aprendizaje.

**Referencias:** Tesseract OCR; Faster-Whisper (SYSTRAN); EmbeddingGemma 2 (Google/Ollama); PolyAI MINDS-14 es-ES, CC BY 4.0. Fotos: El Tiempo, créditos pendientes de curaduría por imagen. SECOP: documentos públicos, evita difundir datos personales.

---

# FASE 2 · Las mismas funciones, ahora sobre 122 fuentes

**No abras otro cuaderno.** Este bloque sigue al anterior: primero **aprendiste cada motor con un ejemplo**, ahora **escribes bucles de Python** y cambias parámetros para procesar un corpus mayor.

- **62 fotografías** = 30 de `datos/raw/fotos` + 32 de `datos/raw/fotos_catalogadas`.
- **30 actas SECOP** = 30 PDF originales + 30 PNG derivados de su primera página; se cuentan **30 documentos**, no 60.
- **30 voces humanas** = descargas autorizadas PolyAI MINDS-14 (CC BY 4.0), convertidas a PCM16 sin reemplazar originales.
- **122 fuentes** = 62 fotos + 30 actas + 30 voces. No implica 122 noticias ni vínculos entre fuentes.

Durante la **grabación de 60 minutos** muestra los resultados de muestras pequeñas; fuera del video cambia `N_ACTAS=5 → 30`, `N_FOTOS=8 → 62`, `N_VISION=8 → 62` y `N_ASR=3 → 30`. Las descargas e inferencias CPU sobre todo el lote pueden tardar y no necesitan bloquear la narración.

**Para inspeccionar cada fuente:** abre la pestaña del visor del Codespace en el puerto privado 8501. Allí puedes avanzar y retroceder entre archivos, ver el PDF entero junto a su PNG, abrir fotos y escuchar WAV.


### Presupuesto de cómputo: no quemar Codespaces ni GitHub Actions

El principio es **explicar primero, medir después y escalar solo cuando aporte aprendizaje**. Las inferencias reales con CPU no deben ejecutarse automáticamente después de cualquier edición.

| Parámetro editable | Clase | Práctica completa | Coste que debes vigilar |
|---|---:|---:|---|
| `N_ACTAS` | 5 | 30 | CPU de OCR |
| `N_FOTOS` | 8 | 62 | CPU de OCR |
| `N_VISION` | 8 | 62 | CPU de embeddings (más costoso) |
| `N_ASR` | 3 | 30 | CPU de Whisper |

**Las pruebas automáticas** deben ser ligeras: sintaxis, conteos, controles de archivos, OCR de muestra y API simulada. **Embeddings reales de decenas de imágenes y ASR en lote se reservan para ejecución manual, justificada y con caché**.

**Reglas de experimentación:** conserva modelo, CPU, versión y parámetros al comparar tiempos; distingue medición real de estimación; registra resultados erróneos y evita afirmar que más tiempo significa mejor calidad.

**EDITA AQUÍ:** comienza por 1, 5 u 8 según el bloque. Escala solo después de verificar que el resultado anterior tiene sentido.

## 1 · Construir el inventario

Aprende a recorrer carpetas con Path y a diferenciar originales de derivados. Los 30 PNG son vistas de los PDF: no cuentan como 30 actas adicionales. Calcula la huella SHA-256 y compara registros distintos por nombre con archivos idénticos por bytes.

In [ ]:
from pathlib import Path
from PIL import Image
from IPython.display import display, Audio
import pandas as pd
import hashlib, json, re, time, io, math

BASE = Path.cwd()
FOTOS_LOTE = sorted((BASE/"datos/raw/fotos").glob("eltiempo_*"))
FOTOS_FICHA = sorted((BASE/"datos/raw/fotos_catalogadas").glob("*.webp"))
ACTAS_PDF = sorted((BASE/"datos/raw/actas_pdf").glob("*.pdf"))
# El primer ejemplo del lote es el mismo contrato investigado en la FASE 1.
CASO_PDF = BASE / "datos/raw/actas_pdf" / CONFIG_S11["acta_caso"]
assert CASO_PDF in ACTAS_PDF, "Falta el caso contractual de la configuración"
ACTAS_PDF = [CASO_PDF] + [p for p in ACTAS_PDF if p != CASO_PDF]
PAGINAS = BASE/"datos/raw/actas"
OUT = BASE/"outputs"
OUT.mkdir(exist_ok=True)

def sha256(ruta):
    h = hashlib.sha256()
    with Path(ruta).open("rb") as f:
        for bloque in iter(lambda: f.read(1024*1024), b""):
            h.update(bloque)
    return h.hexdigest()

print("Fotos:", len(FOTOS_LOTE), "+", len(FOTOS_FICHA))
print("Actas:", len(ACTAS_PDF), "PDF y", len(list(PAGINAS.glob("*.png"))), "PNG")
assert (len(FOTOS_LOTE), len(FOTOS_FICHA), len(ACTAS_PDF)) == (30, 32, 30)
fuentes = FOTOS_LOTE + FOTOS_FICHA + ACTAS_PDF
inv = pd.DataFrame([{"ruta":str(p.relative_to(BASE)), "sha256":sha256(p),
                     "mb":round(p.stat().st_size/1e6,3)} for p in fuentes])
print("Registros:",len(inv),"Huellas distintas:",inv.sha256.nunique())
print("Las rutas de FASE 1 siguen siendo Path; estas listas tienen nombres distintos.")
display(inv.head())

## 2 · OCR documental sobre 5 o 30 actas

Tesseract reconoce caracteres de la primera página PNG; conservamos el PDF completo como documento fuente. Mide tiempo por documento y si se reconocieron palabras como TOTAL o PAGAR. **Detectar una palabra no es medir exactitud**.

En el video usa N_ACTAS=1 (el acta 728704921), y amplía a 5 o 30 únicamente por decisión del estudiante. No compartas OCR completo de documentos con identificaciones.

In [ ]:
import pytesseract
N_ACTAS = CONFIG_S11["limites_demostracion"]["actas"]  # EDITA AQUÍ: 1→5→30 manual
PSM_ACTAS = 3 # EDITA AQUÍ: PSM 3 distribuye bloques; compáralo con PSM 6

filas_actas = []
for pdf in ACTAS_PDF[:N_ACTAS]:
    png = PAGINAS / (pdf.stem + "_pagina1.png")
    if not png.exists():
        raise FileNotFoundError("Sin imagen de primera página: " + pdf.name)
    inicio = time.perf_counter()
    with Image.open(png) as pagina_acta:
        texto = pytesseract.image_to_string(
            pagina_acta, lang="spa", config=f"--psm {PSM_ACTAS}")
    mayusculas = texto.upper()
    filas_actas.append({
        "pdf": pdf.name,
        "sha256_pdf": sha256(pdf),
        "segundos": round(time.perf_counter() - inicio, 2),
        "caracteres": len(texto),
        "detecta_valor_contrato": "VALOR" in mayusculas and "CONTRATO" in mayusculas,
        "detecta_duracion": "DURACI" in mayusculas,
        "detecta_forma_pago": "PAGO" in mayusculas
    })
tabla_actas = pd.DataFrame(filas_actas)
display(tabla_actas)
print("Actas:", len(tabla_actas), "de", len(ACTAS_PDF))
print("Segundos totales medidos:", round(tabla_actas.segundos.sum(), 2))
print("Aviso: detectar etiquetas no mide la exactitud del OCR.")

## 3 · OCR sobre 8 o 62 fotografías

Muchas fotos tienen objetos pero no palabras. Aplica OCR y cuenta caracteres; revisa manualmente falsos positivos en el visor. Elige N_FOTOS=8 para la práctica guiada y 62 solo fuera de la videoclase. La muestra contiene explícitamente los archivos 006, 009 y 023, no las primeras ocho rutas alfabéticas. La longitud de la salida NO demuestra su calidad.

In [ ]:
# La misma muestra de configuracion_s11.json que usa el visor.
FOTOS_COMPLETAS = FOTOS_LOTE + FOTOS_FICHA
por_nombre = {p.name: p for p in FOTOS_COMPLETAS}
assert len(por_nombre) == 62, "Nombres repetidos en el corpus visual"
MUESTRA = [por_nombre[n] for n in CONFIG_S11["muestras_fotografias"]]
TODAS_FOTOS = MUESTRA + [p for p in FOTOS_COMPLETAS if p not in MUESTRA]
assert len(TODAS_FOTOS) == 62 and len(set(TODAS_FOTOS)) == 62
print("Muestra reproducible:", [p.name for p in TODAS_FOTOS[:8]])
N_FOTOS = CONFIG_S11["limites_demostracion"]["fotos"]  # EDITA AQUÍ: 4→8→62 manual
PSM_LOTE_FOTOS = 11  # EDITA AQUÍ: compara con PSM 6

filas_fotos = []
for foto in TODAS_FOTOS[:N_FOTOS]:
    inicio = time.perf_counter()
    with Image.open(foto) as imagen:
        texto = pytesseract.image_to_string(
            imagen, lang="spa", config=f"--psm {PSM_LOTE_FOTOS}")
    filas_fotos.append({
        "archivo": str(foto.relative_to(BASE)),
        "caracteres": len(texto.strip()),
        "detecta_valle": "VALLE" in texto.upper(),
        "muestra_ocr": texto.replace("\n", " ")[:100],
        "segundos": round(time.perf_counter() - inicio, 2)
    })
tabla_fotos = pd.DataFrame(filas_fotos)
display(tabla_fotos.sort_values("caracteres", ascending=False).head(8))
print("Fotos procesadas:", len(tabla_fotos), "de", len(TODAS_FOTOS))
print("Un OCR largo no es necesariamente un OCR correcto: revisa la fuente.")

## 4 · Escalar a 8 y después 62 imágenes sin esconder errores

**Primero comprueba UNA fotografía**, después procesa 4 u 8 y finalmente 62 solo si hay cuota y una hipótesis. La solicitud reproducible para el laboratorio es `{"model": MODELO, "input": [entrada]}`. **Los corchetes exteriores hacen que la imagen viaje como elemento de una lista**. Opcionalmente normalizamos las fotografías a JPEG RGB en memoria con Pillow; esa transformación sirve para comparar formatos, **no para cambiar el formato de la API**. Esto no cambia las fuentes ni garantiza el éxito de cualquier inferencia.

**Explica y edita:** `embedding` siempre usa `"input": [entrada]`. `NORMALIZAR_JPEG = False` utiliza los bytes originales que funcionaron en Codespaces; con `True` convierte temporalmente a JPEG RGB. `N_VISION` controla cuántas fotos se procesan; `MAX_LADO` afecta solo a la transformación JPEG; `imagen_para_modelo` construye Base64. Los errores incluyen el cuerpo HTTP. `cache` conserva vectores con SHA-256, modelo y parámetros de conversión.

**Tu experimento:** 1) prueba un archivo; 2) amplía a ocho; 3) consulta «viviendas en una ladera»; 4) verifica los primeros resultados abriendo las fotos; 5) cambia a 62 fuera de la clase.

In [ ]:
import requests, base64
from PIL import ImageOps
from io import BytesIO

MODELO = CONFIG_S11["modelo_visual"]  # coincide con visor y guía
API = "http://127.0.0.1:11434/api/embed"
MAX_LADO = 768  # EDITA AQUÍ: reduce a 512 solo si NORMALIZAR_JPEG = True

NORMALIZAR_JPEG = False  # EDITA AQUÍ: False = bytes originales (probado); True = JPEG uniforme

def imagen_para_modelo(ruta):
    """Base64 de bytes originales; opcionalmente convierte a JPEG RGB para experimentar."""
    if not NORMALIZAR_JPEG:
        return base64.b64encode(ruta.read_bytes()).decode("ascii")
    with Image.open(ruta) as origen:
        imagen = ImageOps.exif_transpose(origen).convert("RGB")
        imagen.thumbnail((MAX_LADO, MAX_LADO))
        memoria = BytesIO()
        imagen.save(memoria, format="JPEG", quality=85)
    return base64.b64encode(memoria.getvalue()).decode("ascii")

def embedding(entrada, archivo="consulta"):
    """Envía imagen o texto como lista input=[entrada], como se comprobó en Codespaces."""
    try:
        r = requests.post(API, json={"model": MODELO, "input": [entrada]}, timeout=240)
    except requests.RequestException as exc:
        raise RuntimeError("Ollama no responde; verifica bash iniciar_ollama.sh") from exc
    if not r.ok:
        raise RuntimeError(f"Archivo {archivo}, HTTP {r.status_code}: "
                           f"{r.text[:1500]}")
    resultado = r.json().get("embeddings", [])
    if len(resultado) != 1 or len(resultado[0]) != 768:
        raise RuntimeError(f"{archivo}: salida incorrecta; se esperaban 768 valores")
    return resultado[0]

# Una inferencia real por defecto; amplía a 4/8/62 solo bajo demanda.
N_VISION = 1  # EDITA AQUÍ: 1→4→8→62; nunca es automático
assert 1 <= N_VISION <= len(TODAS_FOTOS)
cache = OUT / "vectores_440m.json"
vectores = json.loads(cache.read_text(encoding="utf-8")) if cache.exists() else {}
# Reutiliza las 4 inferencias de la FASE 1 si el modelo y los bytes son iguales.
# No son embeddings de OCR: son vectores de imagen generados realmente en esa fase.
if not NORMALIZAR_JPEG and "vectores_fotos" in globals():
    for nombre, vec in vectores_fotos.items():
        foto_guia = FOTOS / nombre
        if foto_guia.is_file() and len(vec) == 768:
            ruta_rel = str(foto_guia.relative_to(BASE))
            firma = f"{MODELO}|{MAX_LADO}|RAW|{sha256(foto_guia)}"
            anterior = vectores.get(ruta_rel)
            if anterior is None or anterior.get("clave_cache") != firma:
                vectores[ruta_rel] = {"sha256": sha256(foto_guia),
                                    "clave_cache": firma, "vector": vec}
for numero, foto in enumerate(TODAS_FOTOS[:N_VISION], 1):
    key = str(foto.relative_to(BASE))
    huella = sha256(foto)
    clave_cache = f"{MODELO}|{MAX_LADO}|{'JPEG85' if NORMALIZAR_JPEG else 'RAW'}|{huella}"
    if key in vectores and vectores[key].get("clave_cache") == clave_cache:
        print(f"{numero}/{N_VISION}: caché válida, sin calcular de nuevo:", foto.name)
        continue
    print(f"{numero}/{N_VISION}: inferencia real:", foto.name)
    resultado = embedding({"image": imagen_para_modelo(foto)}, key)
    vectores[key] = {"sha256": huella, "clave_cache": clave_cache, "vector": resultado}
    # Guardar después de CADA fotografía: permite retomar tras un error.
    temporal = cache.with_suffix(".tmp")
    temporal.write_text(json.dumps(vectores), encoding="utf-8")
    temporal.replace(cache)  # Guardado atómico, sin JSON incompleto
# Guarda también los vectores heredados de la FASE 1 si no hubo nuevas llamadas.
temporal = cache.with_suffix(".tmp")
temporal.write_text(json.dumps(vectores), encoding="utf-8")
temporal.replace(cache)
for foto in TODAS_FOTOS[:N_VISION]:
    assert len(vectores[str(foto.relative_to(BASE))]["vector"]) == 768
print("Fotografías listas para la consulta:", N_VISION)

In [ ]:
# La búsqueda debe comparar SOLO las fotos del lote actual y la configuración vigente.
# Si la caché contiene pruebas antiguas (otro tamaño, modelo o tratamiento),
# NO debemos incluirlas en este ranking: las comparaciones serían inválidas.
def coseno(a, b):
    num = sum(x*y for x, y in zip(a, b))
    norma_a = math.sqrt(sum(x*x for x in a))
    norma_b = math.sqrt(sum(y*y for y in b))
    if norma_a == 0 or norma_b == 0:
        raise ValueError("No se pueden comparar vectores de norma cero")
    return num/(norma_a*norma_b)

pregunta = "viviendas en una ladera"  # EDITA AQUÍ: lotería, medicamentos, viviendas
consulta = embedding("task: search result | query: " + pregunta, "consulta")
resultados = []
for foto in TODAS_FOTOS[:N_VISION]:
    key = str(foto.relative_to(BASE))
    huella = sha256(foto)
    clave_cache = f"{MODELO}|{MAX_LADO}|{'JPEG85' if NORMALIZAR_JPEG else 'RAW'}|{huella}"
    fila = vectores.get(key)
    if fila is None or fila.get("clave_cache") != clave_cache:
        raise RuntimeError(f"No existe el vector actualizado de {key}; repite la celda anterior")
    resultados.append((key, coseno(consulta, fila["vector"])))

ranking = sorted(resultados, key=lambda x: -x[1])
display(pd.DataFrame(ranking[:10], columns=["archivo", "similitud coseno"]))
print(f"Resultados actuales: {len(ranking)} sobre N_VISION={N_VISION}")
display(Image.open(BASE / ranking[0][0]))

## 5 · Descargar 30 voces humanas con permiso y referencia

Fuente PolyAI MINDS-14 es-ES, licencia CC BY 4.0. Recupera el Parquet real (~39 MB) desde Hugging Face; verifica su SHA-256 antes de leer. Elige una voz de cada una de varias intenciones seleccionadas por rondas. Estos audios bancarios NO son relatos de las fotografías.

In [ ]:
from urllib.request import Request, urlopen
URL = ("https://huggingface.co/datasets/PolyAI/minds14/resolve/main/"
       "es-ES/train-00000-of-00001.parquet")
SHA_ESPERADO = "989bf2f90676a5eb5852fe785a03b1d9b97763e0febbd0fd0a3f489afd83badf"
corpus = BASE/"datos/minds14_es-ES.parquet"
if not corpus.exists():
    temporal = corpus.with_suffix(".part")
    peticion = Request(URL, headers={"User-Agent":"BigData-UCentral-educativo/1.0"})
    try:
        with urlopen(peticion, timeout=180) as red, temporal.open("wb") as salida:
            while bloque := red.read(1024*1024):
                salida.write(bloque)
                if salida.tell() > 100_000_000:
                    raise ValueError("Descarga superior a 100 MB")
        if sha256(temporal) != SHA_ESPERADO:
            raise ValueError("Hash distinto del corpus auditado")
        temporal.replace(corpus)
    finally:
        temporal.unlink(missing_ok=True)
assert sha256(corpus) == SHA_ESPERADO, "Fuente MINDS-14 diferente"
tabla = pd.read_parquet(corpus, columns=["audio","path","transcription","intent_class"])
ordenadas = tabla.sort_values("path", kind="stable").copy()
ordenadas["ronda"] = ordenadas.groupby("intent_class").cumcount()
elegidas = ordenadas.sort_values(["ronda", "intent_class", "path"],
                                 kind="stable").head(30)
assert len(elegidas) == 30 and elegidas["path"].nunique() == 30
print("30 voces reales seleccionadas por rondas reproducibles de intención")
display(elegidas[["path", "intent_class"]].head())

## 6 · Conservar los originales y convertir a WAV PCM16

Un WAV puede contener códecs distintos. SoundFile decodifica la señal y wave guarda una copia PCM16 mono compatible con el reproductor. Conserva hash de fuente y derivado. Después abre cada archivo en el visor.

In [ ]:
import soundfile as sf
import numpy as np
import wave

RAW_AUDIO=BASE/"datos/raw/audios"
DER_AUDIO=BASE/"datos/derived/audios"
RAW_AUDIO.mkdir(parents=True,exist_ok=True)
DER_AUDIO.mkdir(parents=True,exist_ok=True)
voces=[]
for n,(_,fila) in enumerate(elegidas.iterrows(),1):
    original=bytes(fila["audio"]["bytes"])
    fuente=RAW_AUDIO/f"voz_{n:02}_fuente.wav"
    if not fuente.exists(): fuente.write_bytes(original)
    assert sha256(fuente)==hashlib.sha256(original).hexdigest()
    with sf.SoundFile(io.BytesIO(original)) as dec:
        tasa=dec.samplerate
        samples=dec.read(dtype="float32",always_2d=True)
    mono=samples.mean(axis=1)
    pcm=(np.clip(mono,-1,1)*32767).astype("<i2")
    destino=DER_AUDIO/f"voz_{n:02}.wav"
    with wave.open(str(destino),"wb") as archivo_audio:
        archivo_audio.setnchannels(1);archivo_audio.setsampwidth(2);archivo_audio.setframerate(tasa)
        archivo_audio.writeframes(pcm.tobytes())
    voces.append({
        "registro":str(fila["path"]),"intencion":int(fila["intent_class"]),
        "referencia":str(fila["transcription"]),
        "fuente":str(fuente.relative_to(BASE)),"pcm16":str(destino.relative_to(BASE)),
        "sha_fuente":sha256(fuente),"sha_pcm16":sha256(destino)
    })
manifest=BASE/"datos/manifiestos/audios_minds14.json"
manifest.parent.mkdir(parents=True,exist_ok=True)
manifest.write_text(json.dumps({"dataset":"PolyAI/minds14","subset":"es-ES",
    "license":"CC BY 4.0","source_url":URL,"sha256_parquet":SHA_ESPERADO,
    "audios":voces},ensure_ascii=False,indent=2),encoding="utf-8")
display(Audio(filename=str(BASE/voces[0]["pcm16"])))
print("Audios disponibles en el visor:",len(voces))

## 7 · Medir la tasa de error de palabras con Whisper

Whisper tiny ejecuta ASR en CPU con cuantización INT8. La transcripción humana de MINDS-14 sirve como referencia del mismo registro. WER = inserciones + borrados + sustituciones / palabras de referencia. Compara errores y tiempo de tres voces en video y diez para el reto.

In [ ]:
from faster_whisper import WhisperModel

# Reutiliza el modelo ya cargado al inicio si se ejecutó el cuaderno en orden.
if "modelo_voz" in globals():
    whisper = modelo_voz
else:
    whisper = WhisperModel("tiny", device="cpu", compute_type="int8", cpu_threads=2)

def palabras(t):
    return re.findall(r"\w+", t.lower())

def errores(a, b):
    anterior = list(range(len(b) + 1))
    for i, palabra in enumerate(a, 1):
        actual = [i]
        for j, otra in enumerate(b, 1):
            actual.append(min(actual[-1] + 1, anterior[j] + 1,
                              anterior[j-1] + (palabra != otra)))
        anterior = actual
    return anterior[-1]

N_ASR = CONFIG_S11["limites_demostracion"]["audios"]  # EDITA AQUÍ: 1→3→10, bajo demanda
salidas = []
for voz in voces[:N_ASR]:
    ruta = BASE / voz["pcm16"]
    inicio = time.perf_counter()
    segmentos, _ = whisper.transcribe(str(ruta), language="es", beam_size=2)
    prediccion = " ".join(s.text.strip() for s in segmentos)
    ref = palabras(voz["referencia"])
    if not ref:
        raise ValueError("Referencia humana vacía: " + ruta.name)
    salidas.append({
        "archivo": ruta.name,
        "intencion": voz["intencion"],
        "palabras_ref": len(ref),
        "wer": round(errores(ref, palabras(prediccion))/len(ref), 3),
        "segundos": round(time.perf_counter()-inicio, 2),
        "referencia": voz["referencia"][:110],
        "prediccion": prediccion[:110]
    })
tabla_wer = pd.DataFrame(salidas)
display(tabla_wer.sort_values("wer", ascending=False))
print("Voces transcritas:", len(tabla_wer),
      "| tiempo medido:", round(tabla_wer.segundos.sum(), 2), "s")

## 8 · Crear y auditar el catálogo de 122 fuentes

Hay 62 fotos, 30 actas y 30 voces, pero NO existen vínculos documentados entre estas tres colecciones. En la ETL preserva ruta, modalidad, SHA-256, crédito y la relación PDF–PNG. Los registros de las 32 fotografías con ficha tienen metadatos; para las otras 30 no inventes autor ni licencia. Evita exportar el OCR completo de actas SECOP.

In [ ]:
# Un esquema uniforme permite cotejar JSONL y Parquet campo por campo.
# Las vistas PNG y PCM16 son DERIVADOS de una fuente, no nuevas fuentes.
info = json.loads((BASE / "datos/manifiestos/fotos_catalogadas_32.json")
                  .read_text(encoding="utf-8"))
fichas = {Path(x["input_path"]).name: x for x in info["registros"]}
columnas = ["ruta", "tipo", "sha256", "vista", "sha_vista",
            "credito", "url_fuente", "observacion"]
catalogo = []

for foto in TODAS_FOTOS:
    meta = fichas.get(foto.name, {})
    catalogo.append({
        "ruta": str(foto.relative_to(BASE)), "tipo": "foto",
        "sha256": sha256(foto), "vista": "", "sha_vista": "",
        "credito": meta.get("credit") or "pendiente de verificar",
        "url_fuente": meta.get("source_url") or "",
        "observacion": "sin atribuir licencias no verificadas"
    })

for pdf in ACTAS_PDF:
    pagina_png = PAGINAS / (pdf.stem + "_pagina1.png")
    if not pagina_png.is_file():
        raise FileNotFoundError(f"Falta el PNG derivado de {pdf.name}")
    catalogo.append({
        "ruta": str(pdf.relative_to(BASE)), "tipo": "acta",
        "sha256": sha256(pdf),
        "vista": str(pagina_png.relative_to(BASE)),
        "sha_vista": sha256(pagina_png),
        "credito": "SECOP; verificar fuente original",
        "url_fuente": "",
        "observacion": "evitar publicación del OCR con datos personales"
    })

for voz in voces:
    original = BASE / voz["fuente"]
    derivado = BASE / voz["pcm16"]
    catalogo.append({
        "ruta": voz["fuente"], "tipo": "audio",
        "sha256": sha256(original), "vista": voz["pcm16"],
        "sha_vista": sha256(derivado),
        "credito": "PolyAI MINDS-14 es-ES, CC BY 4.0",
        "url_fuente": URL,
        "observacion": "voz bancaria sin relación demostrada con fotos o actas"
    })

assert len(catalogo) == 122
assert all(set(fila) == set(columnas) for fila in catalogo)
archivo = OUT / "catalogo_122.jsonl"
with archivo.open("w", encoding="utf-8") as salida:
    for fila in catalogo:
        salida.write(json.dumps(fila, ensure_ascii=False) + "\n")
pd.DataFrame(catalogo, columns=columnas).to_parquet(
    OUT / "catalogo_122.parquet", index=False)
display(pd.DataFrame(catalogo).groupby("tipo").size())
print("122 fuentes: originales y vistas derivados unidos por sus rutas.")

In [ ]:
# Auditoría integral: misma cantidad, mismos campos, mismos valores, mismos bytes.
jsonl = [json.loads(linea) for linea in archivo.read_text(encoding="utf-8")
         .splitlines() if linea.strip()]
parquet = pd.read_parquet(OUT / "catalogo_122.parquet")
desde_parquet = parquet[columnas].to_dict(orient="records")
assert jsonl == desde_parquet, "El contenido de JSONL y Parquet no es equivalente"
assert len(jsonl) == 122
tipos = pd.Series([x["tipo"] for x in jsonl]).value_counts().to_dict()
assert tipos == {"foto": 62, "acta": 30, "audio": 30}, tipos
assert len({x["ruta"] for x in jsonl}) == 122, "Rutas duplicadas"
for fila in jsonl:
    origen = BASE / fila["ruta"]
    assert origen.is_file() and sha256(origen) == fila["sha256"], fila["ruta"]
    if fila["vista"]:
        derivado = BASE / fila["vista"]
        assert derivado.is_file() and sha256(derivado) == fila["sha_vista"], fila["vista"]
print("OK: JSONL=Parquet; 122 fuentes y derivados auditados.")

# Comprobación independiente: modificar una COPIA de prueba, nunca un original.
copia = OUT / "copia_para_probar_sha.bin"
copia.write_bytes(b"ejemplo A")
antes = sha256(copia)
copia.write_bytes(b"ejemplo B")
despues = sha256(copia)
copia.unlink()
assert antes != despues
print("OK: SHA-256 detectó el cambio de bytes en una copia.")

### Auditoría completa: del resultado a la evidencia original

Una fila útil del catálogo debe permitir **volver al archivo original**, comprobar que está intacto mediante SHA-256 y abrir los derivados asociados.

**No cuentes derivados como nuevas fuentes:** 62 fotografías + 30 PDF SECOP + 30 voces = **122 originales**. Las treinta primeras páginas PNG y los treinta WAV PCM16 son representaciones transformadas de esas fuentes.

**Preguntas de control:**
- ¿El JSONL y el Parquet tienen **los mismos 122 registros y campos**?
- ¿Las rutas originales y los derivados realmente existen?
- ¿Los hashes coinciden con los bytes actuales?
- ¿La atribución y licencia están documentadas o explícitamente pendientes?
- ¿La predicción OCR o ASR tiene referencia humana para comprobarla?

**Integridad no es veracidad.** La prueba de cambiar una **copia temporal** sirve para demostrar que SHA-256 detecta diferencias sin tocar RAW.

**Producto final:** una tabla «pregunta | archivo | herramienta | código editado | resultado | verificación | límite» y una explicación reproducible del experimento. La ejecución sin interpretación no cumple el objetivo.

## Reto final de investigación

Empieza con N_ACTAS=1, N_FOTOS=4, N_VISION=1 y N_ASR=1; comprueba la evidencia. Luego amplía de manera opcional y consciente a 5/8/8/3. Solo si tienes recursos disponibles prueba 30 actas, 62 fotografías o 10 voces: no son requisitos para el video. Compara tiempos medidos, errores y resultados con la misma configuración. Elegir un archivo en el visor y contrastarlo con el resultado. Entregar una tabla: pregunta | corpus | número de archivos | cambio en el código | resultado | evidencia | limitación.

**No es aprendizaje ejecutar todas las celdas sin entenderlas.** Demuestra cómo construiste y modificaste los bucles y cuándo elegiste OCR, búsqueda visual o ASR.

**Abrir visor privado** desde la terminal: streamlit run visor_fuentes.py --server.address 0.0.0.0 --server.port 8501